# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. **A schema** — including the name of the tool, a description, and/or argument definitions (often a JSON schema)

2. **A function or coroutine** — to execute

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why does Bumble bee can't fly?")
response.content

'Actually, the premise of your question is based on a common misconception. **Bumblebees can and do fly.**\n\nThe myth that "bumblebees shouldn\'t be able to fly based on aerodynamics" is a popular piece of folklore, often attributed to a supposed 1930s calculation by a French aerodynamicist or a 1974 conference of the International Congress of Applied Computation. However, **no such formal scientific paper or conference exists**, and the claim has been debunked by entomologists and physicists.\n\n### Why the Myth Exists (and Why It’s Wrong)\n\n1. **Misapplication of Theory**: The myth likely stems from applying simplified, static aerodynamic models (like those used for airplanes) to small, flapping-wing insects. These models don’t account for the complex, unsteady, and three-dimensional nature of insect flight.\n2. **Insect Flight Mechanics**: Bumblebees, like other small insects, use **rotational aerodynamics** and create vortices (like tiny tornadoes) above their wings during each s

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Russia?")
print(response)
for tool_call in response.tool_calls:
    # View tools calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'tool_calls': [{'id': 'my2xgmzbp', 'function': {'arguments': '{"location":"Russia"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.069467371, 'completion_tokens_details': None, 'prompt_time': 0.022099535, 'prompt_tokens_details': None, 'queue_time': 0.048290894, 'total_time': 0.091566906}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0be21-350c-77e3-bff3-56ca62df4ebd-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Russia'}, 'id': 'my2xgmzbp', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Russia'}


### Toll Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{
    "role": "user",
    "content": "What's the weather in Russia?"
}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Russia is 72F and sunny."

It's sunny in Russia! ☀️


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Russia?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kxjy2r9nf', 'function': {'arguments': '{"location":"Russia"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.067529784, 'completion_tokens_details': None, 'prompt_time': 0.01942835, 'prompt_tokens_details': None, 'queue_time': 0.049414, 'total_time': 0.086958134}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0be27-1464-7193-b770-3e9945a34043-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Russia'}, 'id': 'kxjy2r9nf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}),
 ToolMessage(content="It's sunny in R